# Using Z3 as a Theorem Prover

In principle, it should be possible to use Z3 as a theorem prover.  In this notebook, we investigate whether Z3 is able to prove the formula
$$ \forall n \in \mathbb{N}: 2 \cdot \sum\limits_{i=0}^n i = n \cdot (n + 1) $$
by induction.

In [1]:
// load the formatting setup and the style sheet, install Z3 if necessary
#load "../setup.fsx"
installZ3 ()
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
// load Z3
#load "../z3.fsx"
open Microsoft.Z3

We declare an integer variable `n` and an uninterpreted function `Sum` that takes an integer and returns an integer.  The function `Function` is defined in `z3.fsx`.  Its arguments are the name of the function, the list of the sorts of its arguments, and the sort of its result.  A function is applied to an argument `x` via `Sum.Apply(x)`.  As `Apply` returns a general Z3 expression, we use the auxiliary function `sum` that casts the result to an arithmetic expression.

The function `Sum` is specified by the following two axioms:
- $\texttt{Sum}(0) = 0$,
- $\forall n: \texttt{Sum}(n+1) = n + 1 + \texttt{Sum}(n)$.

We add these axioms together with the **negation** of the claim to the solver.  If Z3 is able to show that the resulting set of formulas is unsatisfiable, then the claim has been proven.

In [3]:
let n   = Int "n"
let Sum = Function "Sum" [ intSort ] intSort
let sum (x: Expr) : ArithExpr = Sum.Apply(x) :?> ArithExpr

In [4]:
let S1 = newSolver ()
S1.Add(sum (IntVal 0) .= 0)
S1.Add(ForAll [ n ] (sum (n + 1) .= n + 1 + sum n))
S1.Add(Not (ForAll [ n ] (Implies(n .>= 0, 2 * sum n .= n * (n + 1)))))
S1.Check()

unknown

As we can see, Z3 is not able to prove the claim.  The reason is that a proof requires *induction*, which is not supported by Z3.  Let us check the base case of the induction, where $n = 0$.  Here, `n` is a free variable, i.e. we check whether there is a value of `n` that refutes the claim.

In [5]:
let S2 = newSolver ()
S2.Add(sum (IntVal 0) .= 0)
S2.Add(ForAll [ n ] (sum (n + 1) .= n + 1 + sum n))
S2.Add(Not (Implies(n .= 0, 2 * sum n .= n * (n + 1))))
S2.Check()

unsat

The base case is proven.  However, the general case can not be shown in this way:

In [6]:
let S3 = newSolver ()
S3.Add(sum (IntVal 0) .= 0)
S3.Add(ForAll [ n ] (sum (n + 1) .= n + 1 + sum n))
S3.Add(Not (Implies(n .>= 0, 2 * sum n .= n * (n + 1))))
S3.Check()

unknown

## Group Theory

**Note:** In the Python version of this notebook, the following part refers to the sort `Group`, which is not declared there.  Hence, we declare it here.  The same example is discussed in the notebook `Z3-Group-Theory.ipynb`.

We check whether Z3 is able to prove that the three axioms of a group, which are
* $\mathrm{e} \cdot x = x$
* $x^{-1} \cdot x = \mathrm{e}$
* $(x \cdot y) \cdot z = x \cdot (y \cdot z)$

imply that the theorem 

 * $(x \cdot y)^{-1} = y^{-1} \cdot x^{-1}$

holds.  First, we declare an abstract type `Group`.

In [7]:
let Group = DeclareSort "Group"

Next, we declare the neutral element `e` and three variables `x`, `y`, and `z`.
All of these are elements of the abstract type `Group`.

In [8]:
let e = Const "e" Group
let x = Const "x" Group
let y = Const "y" Group
let z = Const "z" Group

Next, we declare the function `mul` that represents the multiplication of two group elements, i.e. `mul x y` is interpreted as $x \cdot y$
and the function `inv` that represents the inverse of a group element, i.e. `inv x` is interpreted as $x^{-1}$.  The F# functions `mul` and `inv` apply the Z3 functions `mulF` and `invF` to their arguments.

In [9]:
let mulF = Function "mul" [ Group; Group ] Group
let invF = Function "inv" [ Group ] Group
let mul (a: Expr) (b: Expr) : Expr = mulF.Apply(a, b)
let inv (a: Expr) : Expr = invF.Apply(a)

We create a solver and add the axioms to the solver.

In [10]:
let S = newSolver ()
S.Add(ForAll [ x ] (mul e x .= x))
S.Add(ForAll [ x ] (mul (inv x) x .= e))
S.Add(ForAll [ x; y; z ] (mul x (mul y z) .= mul (mul x y) z))

Finally, we add the negation of the theorem that we want to prove.

In [11]:
S.Add(Not (ForAll [ x; y ] (inv (mul x y) .= mul (inv y) (inv x))))

If the theorem is true, then the constraints given to the solver `S` should be unsatisfiable.

In [12]:
#!time
let result = S.Check()

Wall time: 58583.2212ms

In [13]:
match result with
| Status.UNSATISFIABLE -> printfn "The theorem is valid!"
| Status.SATISFIABLE   -> printfn "The theorem is not valid!"
| _                    -> printfn "Z3 is not able to decide whether this theorem is true."

The theorem is valid!